In [1]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

In [2]:
def visualizar_filtro_piel(ruta_imagen):
    print("🔬 CELDA 1: FILTRO DE PIEL")
    img = cv2.imread(str(ruta_imagen))
    if img is None: return print("Error al leer imagen.")

    hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    # Rango HSV para piel humana (en diversas iluminaciones)
    skin_mask = cv2.inRange(hsv, (0, 30, 60), (25, 170, 255))
    skin_pct = np.sum(skin_mask > 0) / skin_mask.size

    paso = skin_pct >= 0.014
    estado = "✅ DETECTADO" if paso else "❌ NO DETECTADO (Posible Logo/Outro)"

    print(f"Porcentaje de piel: {skin_pct*100:.2f}% | Estado: {estado}")

    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    axes[0].imshow(img_rgb)
    axes[0].set_title("Imagen Original")
    axes[0].axis('off')

    axes[1].imshow(skin_mask, cmap='gray')
    axes[1].set_title(f"Máscara de Piel ({skin_pct*100:.2f}%)")
    axes[1].axis('off')
    plt.show()

In [ ]:
def visualizar_filtro_anomalias(ruta_imagen):
    print("🔬 CELDA 2: FILTRO DE ANOMALÍAS (RUIDO)")
    img = cv2.imread(str(ruta_imagen))
    if img is None: return

    h, w = img.shape[:2]
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

    # Recorte del centro de la pantalla
    roi_gray = gray[int(h*0.1):int(h*0.9), int(w*0.28):int(w*0.72)]

    # Limpiamos sombras y extraemos bordes fuertes
    _, roi_limpio = cv2.threshold(roi_gray, 30, 255, cv2.THRESH_TOZERO)
    edges_anom = cv2.Canny(cv2.medianBlur(roi_limpio, 7), 80, 200)

    densidad = np.sum(edges_anom > 0) / edges_anom.size
    paso = densidad <= 0.0495
    estado = "✅ LIMPIO" if paso else "❌ ANOMALÍA DETECTADA (Muy ruidoso)"

    print(f"Densidad de bordes: {densidad*100:.2f}% (Límite: 4.95%) | Estado: {estado}")

    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    axes[0].imshow(roi_limpio, cmap='gray')
    axes[0].set_title("ROI Limpio (Umbral 30)")
    axes[0].axis('off')

    axes[1].imshow(edges_anom, cmap='gray')
    axes[1].set_title(f"Bordes Detectados ({densidad*100:.2f}%)")
    axes[1].axis('off')
    plt.show()

In [ ]:
def visualizar_iconos_y_anillo(ruta_imagen, min_area_icono=2000):
    print("🔬 CELDA 3: ÍCONOS Y CONTRASTE DIFERENCIAL")
    img = cv2.imread(str(ruta_imagen))
    if img is None: return

    h, w = img.shape[:2]
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    # Búsqueda de color y limpieza morfológica
    sat_mask_raw = (hsv[:, :, 1] > 40) & (hsv[:, :, 2] > 80)
    kernel_corte = np.ones((13, 13), np.uint8)
    sat_mask = cv2.morphologyEx(sat_mask_raw.astype(np.uint8), cv2.MORPH_OPEN, kernel_corte)

    contours, _ = cv2.findContours(sat_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    img_resultado = img_rgb.copy()
    validos = 0

    for cnt in contours:
        x, y, cw, ch = cv2.boundingRect(cnt)
        area = cv2.contourArea(cnt)

        if 0.05*min(h,w) < cw < 0.15*min(h,w) and 0.70 < (cw/ch) < 1.35 and area >= min_area_icono:
            margen = 20
            x_out, y_out = max(0, x - margen), max(0, y - margen)
            w_out = min(w - x_out, cw + (margen * 2))
            h_out = min(h - y_out, ch + (margen * 2))

            # Máscara para promediar solo el anillo exterior
            mask_anillo = np.zeros((h, w), dtype=np.uint8)
            cv2.rectangle(mask_anillo, (x_out, y_out), (x_out+w_out, y_out+h_out), 255, -1)
            cv2.rectangle(mask_anillo, (x, y), (x+cw, y+ch), 0, -1)

            brillo_icono = np.mean(gray[y:y+ch, x:x+cw])
            brillo_anillo = cv2.mean(gray, mask=mask_anillo)[0]
            delta = abs(brillo_icono - brillo_anillo)

            es_valido = delta >= 40.0
            color = (0, 255, 0) if es_valido else (255, 0, 0)

            cv2.rectangle(img_resultado, (x_out, y_out), (x_out+w_out, y_out+h_out), color, 2)
            cv2.rectangle(img_resultado, (x, y), (x+cw, y+ch), (0, 255, 255), 2)
            cv2.putText(img_resultado, f"D:{delta:.1f}", (x, y - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.7, color, 2)

            if es_valido: validos += 1
            print(f"Candidato - Delta: {delta:.1f} -> {'✅ APROBADO' if es_valido else '❌ RECHAZADO (Ropa)'}")

    fig, axes = plt.subplots(1, 2, figsize=(15, 6))
    axes[0].imshow(sat_mask, cmap='gray')
    axes[0].set_title("Máscara HSV + Morfología 13x13")
    axes[0].axis('off')

    axes[1].imshow(img_resultado)
    axes[1].set_title(f"Análisis Espacial (Válidos: {validos})")
    axes[1].axis('off')
    plt.show()

In [ ]:
def visualizar_scoring_tiza_oclusion(ruta_imagen, min_area_icono=2000):
    print("🔬 CELDA 4: AISLAMIENTO DE TIZA Y OCLUSIÓN")
    img = cv2.imread(str(ruta_imagen))
    if img is None: return

    h, w = img.shape[:2]
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)

    # Recreamos rápidamente la máscara de piel e íconos para poder medirlos
    skin_mask = cv2.inRange(hsv, (0, 30, 60), (25, 170, 255))
    sat_mask = cv2.morphologyEx(((hsv[:, :, 1] > 40) & (hsv[:, :, 2] > 80)).astype(np.uint8), cv2.MORPH_OPEN, np.ones((13, 13), np.uint8))
    contours, _ = cv2.findContours(sat_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    # Censuramos los íconos válidos en la imagen gris
    gray_masked = gray.copy()
    for cnt in contours:
        x, y, cw, ch = cv2.boundingRect(cnt)
        if cv2.contourArea(cnt) >= min_area_icono:
            cv2.rectangle(gray_masked, (x, y), (x+cw, y+ch), (0,0,0), -1)

    # 1. Medición de Tiza (Centro de la pizarra)
    roi_tiza = gray_masked[int(h*0.1):int(h*0.9), int(w*0.28):int(w*0.72)]
    edges_tiza = cv2.Canny(cv2.GaussianBlur(roi_tiza, (3, 3), 0), 30, 100)
    densidad_tiza = np.sum(edges_tiza > 0) / edges_tiza.size
    tiza_capeada = min(densidad_tiza, 0.015)

    # 2. Medición de Oclusión humana
    roi_skin_center = skin_mask[int(h*0.15):int(h*0.95), int(w*0.28):int(w*0.72)]
    oclusion = np.sum(roi_skin_center > 0) / roi_skin_center.size

    print(f"Tiza detectada: {densidad_tiza*100:.2f}% (Útil para score: {tiza_capeada*100:.2f}%)")
    print(f"Oclusión central (Penalización): {oclusion*100:.2f}%")

    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    axes[0].imshow(gray_masked, cmap='gray')
    axes[0].set_title("1. Escala de Grises (Íconos Censurados)")
    axes[0].axis('off')

    axes[1].imshow(edges_tiza, cmap='gray')
    axes[1].set_title(f"2. Tiza Aislada ({densidad_tiza*100:.2f}%)")
    axes[1].axis('off')

    axes[2].imshow(roi_skin_center, cmap='gray')
    axes[2].set_title(f"3. Oclusión por Humanos ({oclusion*100:.2f}%)")
    axes[2].axis('off')
    plt.show()